In [ ]:
# Document loading
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding
from langchain_community.embeddings import HuggingFaceEmbeddings

# Weaviate
import weaviate
from weaviate.auth import Auth
import weaviate.classes.config as wc

# Ollama LLM
from langchain_ollama import OllamaLLM

# Prompt
from langchain_core.prompts import ChatPromptTemplate

# Memory
from langchain_classic.memory import ConversationBufferMemory

import os

print("All imports done!")

c:\Users\prana\anaconda3\envs\pranayNIT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports done!


In [ ]:
from dotenv import load_dotenv
import os

# Loads variables from .env file
load_dotenv()

# ── Weaviate ──────────────────────────────────────
WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
COLLECTION_NAME = "Documents"

# ── Chunking ──────────────────────────────────────
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

# ── Embedding ─────────────────────────────────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ── Retrieval ─────────────────────────────────────
TOP_K = 5

# ── LLM ───────────────────────────────────────────
OLLAMA_MODEL = "llama3"

# Load PDF
loader = PyPDFLoader("docs/Data Science from Scratch.pdf")
documents = loader.load()

print(f"Loaded {len(documents)} pages")
print(f"Preview: {documents[0].page_content[:200]}")
print("Config ready!")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 71 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 74 0 (offset 0)
Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong

✅ Loaded 330 pages
Preview: DATA/DATA SCIENCE
Data Science from Scratch
ISBN: 978-1-491-90142-7
US $39.99  CAN $45.99
“	Joel	takes	you	on	a	
journey	from	being	
data-curious	to	getting	a	
thorough	understanding	
of	the	bread-and
✅ Config ready!


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)

print(f"✅ Total chunks: {len(chunks)}")
print(f"Sample chunk:\n{chunks[0].page_content}")

✅ Total chunks: 1423
Sample chunk:
DATA/DATA SCIENCE
Data Science from Scratch
ISBN: 978-1-491-90142-7
US $39.99  CAN $45.99
“	Joel	takes	you	on	a	
journey	from	being	
data-curious	to	getting	a	
thorough	understanding	
of	the	bread-and-butter	
algorithms	that	every	data	
scientist 	should 	know.”
—Rohit Sivaprasad
Data Science, Soylent  
datatau.com
Twitter: @oreillymedia
facebook.com/oreilly
Data science libraries, frameworks, modules, and toolkits are great for


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Quick test
test_vector = embeddings.embed_query("hello world")
print(f"Embedding model loaded!")
print(f"Vector size: {len(test_vector)} dimensions")

C:\Users\prana\AppData\Local\Temp\ipykernel_18580\248416190.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


✅ Embedding model loaded!
Vector size: 384 dimensions


In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
)

print(f"Weaviate connected: {client.is_ready()}")

✅ Weaviate connected: True


In [ ]:
if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)
    print("Old collection deleted")

# New API for newer weaviate-client versions
from weaviate.classes.config import Configure, Property, DataType, VectorDistances

client.collections.create(
    name=COLLECTION_NAME,
    vector_config=Configure.Vectors.self_provided(  # we provide our own vectors
        vector_index_config=Configure.VectorIndex.hfresh(
            distance_metric=VectorDistances.COSINE
        )
    ),
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="chunk", data_type=DataType.INT),
    ]
)

print("Collection created!")

✅ Collection created!


In [ ]:
collection = client.collections.get(COLLECTION_NAME)

# Embed all chunks in one batch (fast!)
texts = [chunk.page_content for chunk in chunks]
vectors = embeddings.embed_documents(texts)

print(f"Total vectors to store: {len(vectors)}")

# Store in Weaviate using batch insert
with collection.batch.dynamic() as batch:
    for i, chunk in enumerate(chunks):
        batch.add_object(
            properties={
                "text": chunk.page_content,
                "source": chunk.metadata.get("source", "unknown"),
                "chunk": i,
            },
            vector=vectors[i]
        )

print(f"Stored {len(chunks)} chunks in Weaviate!")

Total vectors to store: 1423
✅ Stored 1423 chunks in Weaviate!


In [ ]:
def retrieve(question, top_k=TOP_K):
    # Embed the question
    query_vector = embeddings.embed_query(question)

    # Hybrid search — vector + keyword combined
    results = collection.query.hybrid(
        query=question,        # BM25 keyword search
        vector=query_vector,   # semantic search
        alpha=0.5,             # 50% vector, 50% keyword
        limit=top_k,
        return_properties=["text", "source", "chunk"]
    )

    return results.objects

# Test it
question = "what is this document about?"
results = retrieve(question)

print(f"Retrieved {len(results)} chunks")
print("\nTop result:")
print(results[0].properties["text"])

✅ Retrieved 5 chunks

Top result:
# not Pythonic
for i in range(len(documents)):
    document = documents[i]
    do_something(i, document)
# also not Pythonic
i = 0
for document in documents:
    do_something(i, document)
    i += 1
The Pythonic solution is enumerate, which produces tuples (index, element):
for i, document in enumerate(documents):
    do_something(i, document)
Similarly, if we just want the indexes:
for i in range(len(documents)): do_something(i)     # not Pythonic


In [ ]:
llm = OllamaLLM(model=OLLAMA_MODEL)

def ask(question, chat_history=""):
    # Retrieve relevant chunks
    results = retrieve(question)

    # Format context from retrieved chunks
    context = "\n\n---\n\n".join(
        r.properties["text"] for r in results
    )

    # Collect sources
    sources = list(set(
        r.properties["source"] for r in results
    ))

    # Anti hallucination prompt
    prompt = f"""You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the provided documents."
Do NOT make up any information.

Previous conversation:
{chat_history}

Context:
{context}

Question: {question}

Answer:"""

    # Call Ollama llama3
    answer = llm.invoke(prompt)

    return answer, sources

# Test it
answer, sources = ask("what is enumerate in python?")
print(f"Answer:\n{answer}")
print(f"\nSources: {sources}")

✅ Answer:
Enumerate in Python is a function that produces tuples (index, element).

📄 Sources: ['docs/Data Science from Scratch.pdf']


In [12]:
# Store chat history as a list
chat_history = []

def chat(question):
    # Format previous conversation into a string
    history_text = "\n".join([
        f"Human: {h['question']}\nAssistant: {h['answer']}"
        for h in chat_history
    ])

    # Get answer with history context
    answer, sources = ask(question, history_text)

    # Save this turn to memory
    chat_history.append({
        "question": question,
        "answer": answer
    })

    print(f" You: {question}")
    print(f" Bot: {answer}")
    print(f" Sources: {sources}")
    print("─" * 50)

    return answer

# Test multi turn conversation
chat("what is enumerate in python?")
chat("can you give me an example of it?")
chat("what is the difference between range and enumerate?")

🧑 You: what is enumerate in python?
🤖 Bot: Enumerate in Python produces tuples (index, element).
📄 Sources: ['docs/Data Science from Scratch.pdf']
──────────────────────────────────────────────────
🧑 You: can you give me an example of it?
🤖 Bot: I don't know based on the provided documents. The context does not contain any information about enumerate in Python. It only talks about networks and friendship relations, with no mention of programming concepts like enumerate.
📄 Sources: ['docs/Data Science from Scratch.pdf']
──────────────────────────────────────────────────


c:\Users\prana\anaconda3\envs\pranayNIT\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
c:\Users\prana\anaconda3\envs\pranayNIT\Lib\ast.py:602: DeprecationWarning: Constant.__init__ got an unexpected keyword argument 's'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  return Constant(*args, **kwargs)
c:\Users\prana\anaconda3\envs\pranayNIT\Lib\ast.py:602: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  return Constant(*args, **kwargs)
c:\Users\prana\anaconda3\envs\pranayNIT\Lib\ast.py:602: DeprecationWarning: Constant.__init__ missing 1 required positional argument: 'value'. This will become an error in Python 3.15.
  return Constant(*args, **kwargs)


KeyboardInterrupt: 